In [1]:
import pandas as pd
from pprint import pprint
#filter to only the rows where the model is resnet50
# gb = cc.groupby(["backbone", "pooling", "sampler", "weight_config", "fulltune"])
coralcam_groups = ['aggression', 'biting']
fishfollow_groups = ['habitat', 'movement', 'bites', 'social_interaction', 'not_visible']

cc = pd.concat([pd.read_csv("../results/eccv/coralcam_eccv_label_tolerance_7.csv"), 
                pd.read_csv("../results/new_grouping/coralcam_results_label_tolerance_7.csv")], ignore_index=True)

ff = pd.concat([pd.read_csv("../results/eccv/fishfollow_eccv_label_tolerance_7.csv"), 
                pd.read_csv("../results/new_grouping/fishfollow_results_label_tolerance_7.csv")], ignore_index=True)

In [2]:
ff['weight_config']

0                          {'weight_method': 'uniform'}
1     {'weight_method': 'focal_loss', 'focal_loss_al...
2                          {'weight_method': 'uniform'}
3     {'weight_method': 'focal_loss', 'focal_loss_al...
4                          {'weight_method': 'uniform'}
5     {'weight_method': 'focal_loss', 'focal_loss_al...
6                          {'weight_method': 'uniform'}
7     {'weight_method': 'focal_loss', 'focal_loss_al...
8                               weight_method='uniform'
9     {'weight_method': 'focal_loss', 'focal_loss_al...
10                         {'weight_method': 'uniform'}
11    {'weight_method': 'focal_loss', 'focal_loss_al...
12                         {'weight_method': 'uniform'}
13    weight_method='focal_loss' focal_loss_gamma=5....
14                              weight_method='uniform'
15    {'weight_method': 'focal_loss', 'focal_loss_al...
16                         {'weight_method': 'uniform'}
17    {'weight_method': 'focal_loss', 'focal_los

In [3]:
import re
def parse_weight_config(weight_config):
    s = str(weight_config)

    # Case 1: plain string (ex: weight_method='focal_loss')
    match1 = re.search(
        r"weight_method\s*[:=]\s*['\"]?([A-Za-z_]+)",
        s
    )

    # Case 2: dict-like (ex: {'weight_method': 'focal_loss', ...})
    match2 = re.search(
        r"'weight_method'\s*:\s*['\"]?([A-Za-z_]+)",
        s
    )

    if match1:
        return match1.group(1)
    if match2:
        return match2.group(1)
    return None

cc['ci_strategy'] = cc['weight_config'].apply(parse_weight_config)
ff['ci_strategy'] = ff['weight_config'].apply(parse_weight_config)

cc.fillna({"freeze_backbone": False}, inplace=True)
cc['fulltune_status'] = cc['fulltune'] & ~cc['freeze_backbone']

ff.fillna({"freeze_backbone": False}, inplace=True)
ff['fulltune_status'] = ff['fulltune'] & ~ff['freeze_backbone']

/tmp/ipykernel_1112304/195439653.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cc.fillna({"freeze_backbone": False}, inplace=True)
/tmp/ipykernel_1112304/195439653.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ff.fillna({"freeze_backbone": False}, inplace=True)


In [4]:
ff.groupby(['backbone', 'ci_strategy']).groups

{('dinov3_base', 'focal_loss'): [1, 3], ('dinov3_base', 'uniform'): [0, 2], ('dinov3_large', 'focal_loss'): [9, 11], ('dinov3_large', 'uniform'): [8, 10], ('resnet50', 'focal_loss'): [17, 25], ('resnet50', 'uniform'): [16, 23, 24], ('resnet50', nan): [26], ('videomae', 'focal_loss'): [18, 22], ('videomae', 'uniform'): [19, 20, 21], ('videomae_large', 'focal_loss'): [5, 7], ('videomae_large', 'uniform'): [4, 6], ('vjepa2', 'focal_loss'): [13, 15], ('vjepa2', 'uniform'): [12, 14]}

In [5]:
ff.columns

Index(['id', 'model', 'epochs', 'dataset', 'monitor', 'pooling', 'sampler',
       'shuffle', 'backbone', 'fulltune',
       ...
       'Idle__Sand habitat_habitat', 'Coral habitat__Sand habitat_habitat',
       'Rubble habitat__Sand habitat_habitat',
       'Aggressive by focal__Sand habitat_habitat',
       'Sand habitat__Sand habitat_habitat', 'aggregator', 'loss_weighted',
       'weight_method', 'ci_strategy', 'fulltune_status'],
      dtype='object', length=166)

In [6]:
def get_full_table(df: pd.DataFrame, group_names, max_metric):
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby(["backbone", 'pooling', 'ci_strategy'])[max_metric].idxmax()]
    return best_rows[["backbone", "pooling", "ci_strategy"] + [max_metric] +
            [f"{group}_{max_metric}" for group in group_names] ]

In [7]:
get_full_table(cc, coralcam_groups, "f1_macro")

,backbone,pooling,ci_strategy,f1_macro,aggression_f1_macro,biting_f1_macro
1,dinov3_base,attention,focal_loss,0.288791,0.000000,0.433186
0,dinov3_base,attention,uniform,0.309183,0.000000,0.463774
3,dinov3_base,mean,focal_loss,0.241447,0.004955,0.359694
2,dinov3_base,mean,uniform,0.025878,0.000000,0.038817
11,dinov3_large,attention,focal_loss,0.329175,0.006579,0.490473
10,dinov3_large,attention,uniform,0.324976,0.000000,0.487464
13,dinov3_large,mean,focal_loss,0.230111,0.001428,0.344453
12,dinov3_large,mean,uniform,0.237616,0.000918,0.355965
19,resnet50,attention,focal_loss,0.285857,0.000000,0.428786
18,resnet50,attention,uniform,0.285276,0.000000,0.427914


In [8]:
get_full_table(cc, coralcam_groups, "mAP")

,backbone,pooling,ci_strategy,mAP,aggression_mAP,biting_mAP
1,dinov3_base,attention,focal_loss,0.295154,0.001124,0.442170
0,dinov3_base,attention,uniform,0.276029,0.001799,0.413143
3,dinov3_base,mean,focal_loss,0.165177,0.001869,0.246830
2,dinov3_base,mean,uniform,0.078043,0.000000,0.117065
11,dinov3_large,attention,focal_loss,0.321502,0.001237,0.481634
10,dinov3_large,attention,uniform,0.315239,0.000819,0.472448
13,dinov3_large,mean,focal_loss,0.151057,0.001448,0.225862
12,dinov3_large,mean,uniform,0.108423,0.001362,0.161953
19,resnet50,attention,focal_loss,0.158666,0.000642,0.237678
18,resnet50,attention,uniform,0.161307,0.000839,0.241541


In [9]:
get_full_table(ff, fishfollow_groups, "f1_macro")

,backbone,pooling,ci_strategy,f1_macro,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro
1,dinov3_base,attention,focal_loss,0.377576,0.582361,0.464945,0.108132,0.055101,0.457178
0,dinov3_base,attention,uniform,0.320052,0.577074,0.444237,0.002041,0.013156,0.188997
3,dinov3_base,mean,focal_loss,0.367592,0.560476,0.464037,0.090770,0.135412,0.369358
2,dinov3_base,mean,uniform,0.317454,0.567811,0.441232,0.000966,0.021258,0.193152
9,dinov3_large,attention,focal_loss,0.379678,0.593821,0.458118,0.123250,0.036066,0.457943
8,dinov3_large,attention,uniform,0.373225,0.605308,0.480125,0.038517,0.070819,0.449004
11,dinov3_large,mean,focal_loss,0.364806,0.565924,0.457688,0.110997,0.035302,0.387980
10,dinov3_large,mean,uniform,0.317467,0.574687,0.432288,0.004938,0.039468,0.187291
17,resnet50,attention,focal_loss,0.360420,0.548184,0.454412,0.102525,0.027916,0.433353
16,resnet50,attention,uniform,0.360611,0.571164,0.460216,0.081677,0.022860,0.405485


In [10]:
get_full_table(ff, fishfollow_groups, "mAP")

,backbone,pooling,ci_strategy,mAP,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
1,dinov3_base,attention,focal_loss,0.304945,0.524114,0.385562,0.004033,0.033348,0.418690
0,dinov3_base,attention,uniform,0.293474,0.499634,0.378864,0.003751,0.032421,0.378264
3,dinov3_base,mean,focal_loss,0.296692,0.492631,0.391672,0.004030,0.050978,0.357673
2,dinov3_base,mean,uniform,0.297594,0.493183,0.392812,0.004293,0.050037,0.362197
9,dinov3_large,attention,focal_loss,0.307225,0.534870,0.384305,0.003993,0.036394,0.419410
8,dinov3_large,attention,uniform,0.313493,0.522433,0.399622,0.003961,0.046354,0.451757
11,dinov3_large,mean,focal_loss,0.299616,0.500942,0.389340,0.004198,0.034183,0.398701
10,dinov3_large,mean,uniform,0.302815,0.502751,0.400867,0.004287,0.048632,0.362520
17,resnet50,attention,focal_loss,0.267906,0.448895,0.348440,0.003594,0.029967,0.353151
16,resnet50,attention,uniform,0.268328,0.440841,0.353104,0.003712,0.029329,0.359760


In [11]:
def get_backbone_table(df: pd.DataFrame, group_names, max_metric):
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby("backbone")[max_metric].idxmax()]
    return best_rows[["backbone"] + ["f1_macro", "mAP"] +
            [f"{group}_f1_macro" for group in group_names] +
            [f"{group}_mAP" for group in group_names]]

In [12]:
get_backbone_table(cc, coralcam_groups, "f1_macro")

,backbone,f1_macro,mAP,aggression_f1_macro,biting_f1_macro,aggression_mAP,biting_mAP
0,dinov3_base,0.309183,0.276029,0.000000,0.463774,0.001799,0.413143
11,dinov3_large,0.329175,0.321502,0.006579,0.490473,0.001237,0.481634
19,resnet50,0.285857,0.158666,0.000000,0.428786,0.000642,0.237678
23,videomae,0.362011,0.130226,0.310870,0.387581,0.063816,0.163431
5,videomae_large,0.450383,0.201312,0.331539,0.509804,0.204558,0.199689
17,vjepa2,0.356212,0.252290,0.294118,0.387260,0.262049,0.247411


In [13]:
get_backbone_table(ff, fishfollow_groups, "f1_macro")

,backbone,f1_macro,mAP,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
1,dinov3_base,0.377576,0.304945,0.582361,0.464945,0.108132,0.055101,0.457178,0.524114,0.385562,0.004033,0.033348,0.418690
9,dinov3_large,0.379678,0.307225,0.593821,0.458118,0.123250,0.036066,0.457943,0.534870,0.384305,0.003993,0.036394,0.419410
16,resnet50,0.360611,0.268328,0.571164,0.460216,0.081677,0.022860,0.405485,0.440841,0.353104,0.003712,0.029329,0.359760
22,videomae,0.370142,0.301287,0.552342,0.453869,0.104541,0.085402,0.486447,0.505345,0.382440,0.004217,0.051644,0.424197
5,videomae_large,0.386816,0.307633,0.562756,0.472224,0.148980,0.121287,0.410989,0.506739,0.402407,0.005599,0.061159,0.389025
15,vjepa2,0.375297,0.334886,0.566341,0.484327,0.118818,0.000000,0.401754,0.551765,0.444073,0.004929,0.031648,0.431417


In [14]:
def get_binary_pivot_table(df: pd.DataFrame, pivot, pivot_values: list, max_metric): 
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby(["backbone", pivot])[max_metric].idxmax()]
    #pivot the table to have ci_strategy as columns and backbone as rows
    pivoted = best_rows.pivot(index="backbone", columns=pivot, values=["f1_macro", "mAP", 'precision_macro', 'recall_macro'])
    assert len(pivot_values) == 2, "Only supports two pivot values for now"
    pivoted["delta_precision_macro"] = (
        pivoted[("precision_macro", pivot_values[0])]
        - pivoted[("precision_macro", pivot_values[1])]
    )
    pivoted["delta_recall_macro"] = (
        pivoted[("recall_macro", pivot_values[0])]
        - pivoted[("recall_macro", pivot_values[1])]
    )
    pivoted["delta_f1_macro"] = (
        pivoted[("f1_macro", pivot_values[0])]
        - pivoted[("f1_macro", pivot_values[1])]
    )
    #delete precision_macro and recall_macro columns
    pivoted = pivoted.drop(
        columns=[("precision_macro", pivot_values[0]), ("precision_macro", pivot_values[1]), ("recall_macro", pivot_values[0]), ("recall_macro", pivot_values[1])])
    return pivoted.reset_index()


In [15]:
get_binary_pivot_table(cc, "ci_strategy", ["focal_loss", "uniform"], "f1_macro")

backbone   f1_macro                  mAP            \
ci_strategy                 focal_loss   uniform focal_loss   uniform   
0               dinov3_base   0.288791  0.309183   0.295154  0.276029   
1              dinov3_large   0.329175  0.324976   0.321502  0.315239   
2                  resnet50   0.285857  0.285276   0.158666  0.161307   
3                  videomae   0.288407  0.362011   0.118630  0.130226   
4            videomae_large   0.450383  0.260826   0.201312  0.083490   
5                    vjepa2   0.356212  0.352122   0.252290  0.234684   

            delta_precision_macro delta_recall_macro delta_f1_macro  
ci_strategy                                                          
0                       -0.021950           0.035774      -0.020392  
1                       -0.000911           0.066890       0.004199  
2                        0.001197           0.010937       0.000581  
3                       -0.098642          -0.074850      -0.073604  
4                        0.225482           0.042335       0.189557  
5                       -0.009901           0.061642       0.004090

In [16]:
get_binary_pivot_table(ff, "ci_strategy", ["focal_loss", "uniform"], "f1_macro")

backbone   f1_macro                  mAP            \
ci_strategy                 focal_loss   uniform focal_loss   uniform   
0               dinov3_base   0.377576  0.320052   0.304945  0.293474   
1              dinov3_large   0.379678  0.373225   0.307225  0.313493   
2                  resnet50   0.360420  0.360611   0.267906  0.268328   
3                  videomae   0.370142  0.357933   0.301287  0.303900   
4            videomae_large   0.386816  0.369281   0.307633  0.306801   
5                    vjepa2   0.375297  0.356103   0.334886  0.348479   

            delta_precision_macro delta_recall_macro delta_f1_macro  
ci_strategy                                                          
0                       -0.041934           0.186035       0.057523  
1                       -0.044236           0.129178       0.006453  
2                       -0.036799           0.166843      -0.000191  
3                       -0.056325           0.154537       0.012208  
4                       -0.034557           0.160629       0.017535  
5                       -0.093010           0.120855       0.019195

In [17]:
get_binary_pivot_table(cc, "pooling", ['attention', 'mean'], "f1_macro")

backbone  f1_macro                                  mAP  \
pooling                 attention      mean vjepa2_attention attention   
0           dinov3_base  0.309183  0.241447              NaN  0.276029   
1          dinov3_large  0.329175  0.237616              NaN  0.321502   
2              resnet50  0.285857  0.249886              NaN  0.158666   
3              videomae  0.362011  0.192891              NaN  0.130226   
4        videomae_large  0.450383  0.243682              NaN  0.201312   
5                vjepa2  0.241273  0.356212         0.342927  0.100342   

                                    precision_macro     recall_macro  \
pooling      mean vjepa2_attention vjepa2_attention vjepa2_attention   
0        0.165177              NaN              NaN              NaN   
1        0.108423              NaN              NaN              NaN   
2        0.107070              NaN              NaN              NaN   
3        0.111989              NaN              NaN              NaN   
4        0.126821              NaN              NaN              NaN   
5        0.252290         0.142878         0.238041          0.70488   

        delta_precision_macro delta_recall_macro delta_f1_macro  
pooling                                                          
0                    0.060198          -0.126344       0.067735  
1                    0.059874           0.155977       0.091558  
2                    0.036827          -0.026562       0.035972  
3                    0.183486          -0.171300       0.169120  
4                    0.221248           0.053215       0.206701  
5                   -0.079213          -0.113792      -0.114939

In [18]:
get_binary_pivot_table(ff, "pooling", ['attention', 'mean'], "f1_macro")

backbone  f1_macro                 mAP            \
pooling                 attention      mean attention      mean   
0           dinov3_base  0.377576  0.367592  0.304945  0.296692   
1          dinov3_large  0.379678  0.364806  0.307225  0.299616   
2              resnet50  0.360611  0.349642  0.268328  0.264370   
3              videomae  0.370142  0.338825  0.301287  0.299193   
4        videomae_large  0.386816  0.354790  0.307633  0.317525   
5                vjepa2  0.371680  0.375297  0.297917  0.334886   

        delta_precision_macro delta_recall_macro delta_f1_macro  
pooling                                                          
0                    0.002317           0.038157       0.009984  
1                    0.005585           0.042149       0.014872  
2                    0.033261          -0.062978       0.010969  
3                    0.006143           0.111556       0.031317  
4                   -0.016458           0.130756       0.032026  
5                   -0.032490           0.079891      -0.003617

In [19]:
def get_resnet_table(df, group_names, max_metric):
    dino = df[df['backbone'] == 'dinov3_large']
    resnet_frozen = df[(df['backbone'] == 'resnet50') & (df['fulltune_status'] == False)]
    resnet_fulltune = df[(df['backbone'] == 'resnet50') & (df['fulltune_status'] == True)]

    dino_best = dino.loc[dino[max_metric].idxmax()]
    resnet_frozen_best = resnet_frozen.loc[resnet_frozen[max_metric].idxmax()]
    resnet_fulltune_best = resnet_fulltune.loc[resnet_fulltune[max_metric].idxmax()]
    
    
    # columns you care about
    cols = (
        ["backbone", "fulltune_status", "f1_macro", "mAP"]
        + [f"{g}_f1_macro" for g in group_names]
        + [f"{g}_mAP" for g in group_names]
    )

    # each *_best is a Series → turn into 1-row DataFrame with .to_frame().T
    table = pd.concat(
        [row[cols].to_frame().T for row in [dino_best, resnet_frozen_best, resnet_fulltune_best]],
        ignore_index=True,
    )

    return table

In [20]:
get_resnet_table(cc, coralcam_groups, "f1_macro")

,backbone,fulltune_status,f1_macro,mAP,aggression_f1_macro,biting_f1_macro,aggression_mAP,biting_mAP
0,dinov3_large,False,0.329175,0.321502,0.006579,0.490473,0.001237,0.481634
1,resnet50,False,0.285857,0.158666,0.0,0.428786,0.000642,0.237678
2,resnet50,True,0.357826,0.242816,0.0,0.53674,0.000642,0.363903


In [21]:
get_resnet_table(ff, fishfollow_groups, "f1_macro")

,backbone,fulltune_status,f1_macro,mAP,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
0,dinov3_large,False,0.379678,0.307225,0.593821,0.458118,0.12325,0.036066,0.457943,0.53487,0.384305,0.003993,0.036394,0.41941
1,resnet50,False,0.360611,0.268328,0.571164,0.460216,0.081677,0.02286,0.405485,0.440841,0.353104,0.003712,0.029329,0.35976
2,resnet50,True,0.321379,0.298217,0.549549,0.40456,0.040642,0.0,0.384554,0.527068,0.380348,0.003259,0.028926,0.355172
